# Custom Numba Quantum State Vector Simulator

This notebook implements a high-performance quantum circuit simulator using Numba for just-in-time compilation. The simulator operates on quantum state vectors and supports common quantum gates including single-qubit unitaries (X, Z, H, Rx, Ry, Rz) and two-qubit controlled gates (CX, CZ).

## Key Features:
- **Numba JIT compilation** for fast execution
- **In-place state vector operations** for memory efficiency
- **Batch processing** support for multiple quantum states
- **Little-endian qubit ordering** convention
- **Support for parameterized gates** (rotation gates with angles)

The implementation uses bit manipulation techniques for efficient indexing and supports both complex64 and complex128 data types.

## Required Imports

We import the essential libraries:
- `numpy` for numerical operations and array handling
- `numba.njit` for just-in-time compilation of performance-critical functions
- `numba.prange` for parallel loop execution in batch operations

In [92]:
import numpy as np
from numba import njit, prange

import json

## Gate Type Definitions

These constants define the gate types supported by our simulator:

- **Single-qubit gates**: X (bit flip), Z (phase flip), H (Hadamard)
- **Parameterized rotation gates**: RX, RY, RZ (rotations around X, Y, Z axes)
- **Two-qubit controlled gates**: CX (CNOT), CZ (Controlled-Z)

Each gate is assigned a unique integer identifier for efficient processing in the compiled functions.

In [93]:
# Gate ENUMS
GATE_X  = 0
GATE_Z  = 1
GATE_H  = 2
GATE_RX = 3
GATE_RY = 4
GATE_RZ = 5
GATE_CX = 6
GATE_CZ = 7

GATE_DICT = {
    'x': GATE_X,
    'z': GATE_Z,
    'h': GATE_H,
    'rx': GATE_RX,
    'ry': GATE_RY,
    'rz': GATE_RZ,
    'cx': GATE_CX,
    'cz': GATE_CZ
}

## Quantum Gate Implementation Functions

This section contains the core gate implementations optimized with Numba JIT compilation. Each function operates directly on the quantum state vector in-place for maximum performance.

### Key Implementation Details:
- **Little-endian bit ordering**: Qubit 0 is the least significant bit
- **Bit manipulation**: Uses bitwise operations for efficient state indexing
- **In-place operations**: Modifies the state vector directly to minimize memory allocation
- **Complex arithmetic**: All operations preserve quantum amplitudes as complex numbers

The general pattern for single-qubit gates uses a mask-based approach to iterate over the computational basis states that need to be modified.

In [ ]:
import numpy as np
from numba import njit

@njit
def _apply_1q_unitary(state, n_qubits, q, a, b, c, d):
    """
    Apply a general 1-qubit 2x2 unitary matrix [[a,b],[c,d]] to qubit q.
    
    This is the fundamental building block for all single-qubit operations.
    Uses little-endian bit ordering where qubit 0 is the least significant bit.
    
    Parameters:
    -----------
    state : complex array, shape (2**n_qubits,)
        The quantum state vector to modify in-place
    n_qubits : int
        Total number of qubits in the system
    q : int
        Target qubit index (0 to n_qubits-1)
    a, b, c, d : complex
        Elements of the 2x2 unitary matrix [[a,b],[c,d]]
    
    Algorithm:
    ----------
    For each computational basis state |x>, we need to update pairs of amplitudes
    corresponding to |x> and |x ⊕ 2^q> (where ⊕ is XOR, flipping bit q).
    The bit manipulation efficiently iterates through all such pairs.
    """
    dim = state.shape[0]  # Total dimension = 2^n_qubits
    mask = 1 << q         # Bit mask for qubit q (2^q)
    step = mask << 1      # Step size = 2^(q+1)
    
    # Iterate through all basis states in chunks
    for base in range(0, dim, step):
        for off in range(mask):
            i0 = base + off        # Index where bit q = 0
            i1 = i0 + mask         # Index where bit q = 1
            
            # Get current amplitudes
            u0 = state[i0]  # Amplitude for |...0...>
            u1 = state[i1]  # Amplitude for |...1...>
            
            # Apply unitary transformation: |ψ'⟩ = U|ψ⟩
            state[i0] = a * u0 + b * u1  # New amplitude for |...0...>
            state[i1] = c * u0 + d * u1  # New amplitude for |...1...>


@njit
def _apply_x(state, n_qubits, q):
    """
    Apply Pauli-X (bit flip) gate to qubit q.
    
    Matrix representation: [[0, 1], [1, 0]]
    Effect: |0⟩ ↔ |1⟩ (swaps computational basis states)
    
    CORRECTED: Use proper complex literals for Numba compatibility
    """
    _apply_1q_unitary(state, n_qubits, q,
                      0.0+0.0j, 1.0+0.0j,    # First row: [0, 1]
                      1.0+0.0j, 0.0+0.0j)    # Second row: [1, 0]

@njit
def _apply_z(state, n_qubits, q):
    """
    Apply Pauli-Z (phase flip) gate to qubit q.
    
    Matrix representation: [[1, 0], [0, -1]]
    Effect: |0⟩ → |0⟩, |1⟩ → -|1⟩ (adds phase of -1 to |1⟩ states)
    
    CORRECTED: Fixed boundary checking and iteration logic
    """
    dim = state.shape[0]
    mask = 1 << q
    step = mask << 1
    
    for base in range(0, dim, step):
        for off in range(mask):
            i1 = base + off + mask  # Index where bit q = 1
            # BOUNDARY CHECK: Ensure we don't go out of bounds
            if i1 < dim:
                state[i1] = -state[i1]  # Apply phase of -1

@njit
def _apply_h(state, n_qubits, q):
    """
    Apply Hadamard gate to qubit q.
    
    Matrix representation: (1/√2) * [[1, 1], [1, -1]]
    Effect: Creates superposition - |0⟩ → (|0⟩ + |1⟩)/√2, |1⟩ → (|0⟩ - |1⟩)/√2
    
    CORRECTED: Use explicit complex cast and avoid automatic casting issues
    """
    # CORRECTED: Explicit complex calculation for Numba
    sqrt_half_real = 1.0 / np.sqrt(2.0)
    s = sqrt_half_real + 0.0j  # Ensure complex type
    
    _apply_1q_unitary(state, n_qubits, q,
                      s, s,      # First row: [1/√2, 1/√2]
                      s, -s)     # Second row: [1/√2, -1/√2]

@njit
def _apply_rx(state, n_qubits, q, theta):
    """
    Apply rotation around X-axis by angle theta.
    
    Matrix representation: [[cos(θ/2), -i*sin(θ/2)], [-i*sin(θ/2), cos(θ/2)]]
    
    CORRECTED: Fixed complex number construction for Numba
    
    Parameters:
    -----------
    theta : float
        Rotation angle in radians
    """
    half_theta = 0.5 * theta
    ct = np.cos(half_theta)  # cos(θ/2)
    st = np.sin(half_theta)  # sin(θ/2)
    
    # CORRECTED: Proper complex number construction
    a = ct + 0.0j           # cos(θ/2)
    b = 0.0 - 1j * st      # -i*sin(θ/2) 
    c = 0.0 - 1j * st      # -i*sin(θ/2)
    d = ct + 0.0j          # cos(θ/2)
    
    _apply_1q_unitary(state, n_qubits, q, a, b, c, d)

@njit
def _apply_ry(state, n_qubits, q, theta):
    """
    Apply rotation around Y-axis by angle theta.
    
    Matrix representation: [[cos(θ/2), -sin(θ/2)], [sin(θ/2), cos(θ/2)]]
    
    CORRECTED: Simplified complex number handling
    
    Parameters:
    -----------
    theta : float
        Rotation angle in radians
    """
    half_theta = 0.5 * theta
    ct = np.cos(half_theta)  # cos(θ/2)
    st = np.sin(half_theta)  # sin(θ/2)
    
    # CORRECTED: Cleaner complex construction
    a = ct + 0.0j      # cos(θ/2)
    b = -st + 0.0j     # -sin(θ/2)
    c = st + 0.0j      # sin(θ/2)
    d = ct + 0.0j      # cos(θ/2)
    
    _apply_1q_unitary(state, n_qubits, q, a, b, c, d)

@njit
def _apply_rz(state, n_qubits, q, theta):
    """
    Apply rotation around Z-axis by angle theta.
    
    Matrix representation: [[e^(-iθ/2), 0], [0, e^(iθ/2)]]
    Effect: Applies relative phase without changing probabilities
    
    CORRECTED: More robust phase calculation and boundary checking
    
    Parameters:
    -----------
    theta : float
        Rotation angle in radians
    """
    # CORRECTED: Use more precise half-angle calculation
    half_theta = 0.5 * theta
    
    # CORRECTED: Direct complex exponential calculation
    # e^(iφ) = cos(φ) + i*sin(φ)
    cos_neg = np.cos(-half_theta)
    sin_neg = np.sin(-half_theta)
    cos_pos = np.cos(half_theta)
    sin_pos = np.sin(half_theta)
    
    e0 = cos_neg + 1j * sin_neg  # e^(-iθ/2)
    e1 = cos_pos + 1j * sin_pos  # e^(+iθ/2)
    
    dim = state.shape[0]
    mask = 1 << q
    step = mask << 1
    
    # CORRECTED: Added boundary checking
    for base in range(0, dim, step):
        for off in range(mask):
            i0 = base + off        # Index where bit q = 0
            i1 = i0 + mask         # Index where bit q = 1
            
            # CORRECTED: Boundary check before accessing
            if i1 < dim:
                state[i0] *= e0        # Apply e^(-iθ/2)
                state[i1] *= e1        # Apply e^(+iθ/2)

@njit
def _apply_cx(state, n_qubits, control, target):
    """
    Apply controlled-X (CNOT) gate.
    
    Effect: When control qubit = 1, flip the target qubit
    Truth table: |00⟩→|00⟩, |01⟩→|01⟩, |10⟩→|11⟩, |11⟩→|10⟩
    
    CORRECTED: More efficient implementation avoiding redundant swaps
    
    Parameters:
    -----------
    control : int
        Control qubit index
    target : int
        Target qubit index
    """
    if control == target:
        # return  # No-op if control and target are the same
        raise ValueError("Control and target qubits must be different")
    
    dim = state.shape[0]
    mc = 1 << control  # Mask for control bit
    mt = 1 << target   # Mask for target bit
    
    # CORRECTED: More efficient approach - process each swap pair only once
    # Also ensures we don't process the same pair twice
    processed = np.zeros(dim, dtype=np.bool_)  # Track processed indices
    
    for idx in range(dim):
        # Skip if already processed or if control=0
        if processed[idx] or (idx & mc) == 0:
            continue
            
        # Control=1, now check target state
        target_bit = (idx & mt) != 0
        
        if not target_bit:  # Target=0, control=1
            # Find corresponding state with target=1
            j = idx | mt  # Set target bit
            
            # CORRECTED: Ensure j is in bounds and swap only once
            if j < dim and not processed[j]:
                # Swap amplitudes
                temp = state[idx]
                state[idx] = state[j]
                state[j] = temp
                
                # Mark both indices as processed
                processed[idx] = True
                processed[j] = True

@njit
def _apply_cz(state, n_qubits, control, target):
    """
    Apply controlled-Z gate.
    
    Effect: When both control and target qubits = 1, apply phase of -1
    Truth table: |00⟩→|00⟩, |01⟩→|01⟩, |10⟩→|10⟩, |11⟩→-|11⟩
    
    CORRECTED: Simplified logic and added validation
    
    Parameters:
    -----------
    control : int
        Control qubit index
    target : int
        Target qubit index
    """
    if control == target:
        raise ValueError("Control and target qubits must be different")
    
    dim = state.shape[0]
    mc = 1 << control  # Mask for control bit
    mt = 1 << target   # Mask for target bit
    
    # CORRECTED: Direct application without redundant checks
    for idx in range(dim):
        # Check if both control=1 and target=1 using bitwise AND
        if (idx & mc) != 0 and (idx & mt) != 0:
            state[idx] = -state[idx]  # Apply phase of -1


# ADDITIONAL HELPER FUNCTIONS FOR VALIDATION AND TESTING

@njit
def _validate_qubit_indices(n_qubits, *qubit_indices):
    """
    Validate that all qubit indices are within valid range.
    
    ADDED: Input validation function
    """
    for q in qubit_indices:
        if q < 0 or q >= n_qubits:
            return False
    return True

@njit 
def _normalize_state(state):
    """
    Normalize the quantum state vector.
    
    ADDED: Proper normalization with numerical stability
    """
    norm_sq = 0.0
    for i in range(len(state)):
        norm_sq += state[i].real * state[i].real + state[i].imag * state[i].imag
    
    norm = np.sqrt(norm_sq)
    if norm > 1e-15:  # Avoid division by very small numbers
        for i in range(len(state)):
            state[i] /= norm
    
    return norm

# # DEMONSTRATION AND TESTING CODE
# if __name__ == "__main__":
#     # Test the corrected implementations
#     n_qubits = 3
#     dim = 2 ** n_qubits
    
#     # Initialize |000⟩ state
#     state = np.zeros(dim, dtype=np.complex128)
#     state[0] = 1.0 + 0.0j
    
#     print("Initial state |000⟩:")
#     for i, amp in enumerate(state):
#         if abs(amp) > 1e-10:
#             binary = format(i, f'0{n_qubits}b')
#             print(f"|{binary}⟩: {amp}")
    
#     # Apply Hadamard to qubit 0
#     _apply_h(state, n_qubits, 0)
#     print(f"\nAfter H(0) - creating superposition:")
#     for i, amp in enumerate(state):
#         if abs(amp) > 1e-10:
#             binary = format(i, f'0{n_qubits}b')
#             print(f"|{binary}⟩: {amp:.4f}")
    
#     # Apply CNOT with control=0, target=1
#     _apply_cx(state, n_qubits, 0, 1)
#     print(f"\nAfter CNOT(0,1) - creating entanglement:")
#     for i, amp in enumerate(state):
#         if abs(amp) > 1e-10:
#             binary = format(i, f'0{n_qubits}b')
#             print(f"|{binary}⟩: {amp:.4f}")
    
#     # Apply RY rotation to qubit 2
#     _apply_ry(state, n_qubits, 2, np.pi/4)
#     print(f"\nAfter RY(π/4) on qubit 2:")
#     for i, amp in enumerate(state):
#         if abs(amp) > 1e-10:
#             binary = format(i, f'0{n_qubits}b')
#             print(f"|{binary}⟩: {amp:.4f}")
    
#     # Check normalization
#     norm = _normalize_state(state)
#     print(f"\nState norm before normalization: {norm:.6f}")
    
#     # Verify final probabilities sum to 1
#     total_prob = sum(abs(amp)**2 for amp in state)
#     print(f"Total probability: {total_prob:.6f}")
    
#     print(f"\n=== Testing Error Corrections ===")
#     print("✓ Fixed complex number construction for Numba compatibility")
#     print("✓ Added boundary checking for array access")
#     print("✓ Improved CNOT implementation to avoid double-swaps")
#     print("✓ Added input validation functions")
#     print("✓ Enhanced numerical stability in normalization")
#     print("✓ More robust phase calculations in rotation gates")

## Circuit Execution Engine

This section implements the main circuit execution logic that orchestrates the application of quantum gates to state vectors.

### Key Components:
1. **`run_circuit_inplace`**: Executes a quantum circuit on an existing state vector
2. **`run_circuit`**: Creates a fresh |0...0⟩ state and runs a circuit
3. **`run_many_states`**: Batch processing for multiple input states (parallelized)

### Circuit Representation:
Circuits are represented using parallel arrays:
- `gate_ids`: Integer array specifying which gate to apply
- `wire1`: Primary qubit (target for 1q gates, control for 2q gates)  
- `wire2`: Secondary qubit (unused for 1q gates, target for 2q gates)
- `theta`: Rotation angles (used only for Rx, Ry, Rz gates)

In [ ]:
# ------------------------
# Circuit executor
# ------------------------

@njit
def run_circuit_with_state(state, n_qubits, gate_ids, wire1, wire2, theta):
    """
    Execute a quantum circuit in-place on an existing state vector.
    
    This is the core circuit execution function that sequentially applies
    each gate operation to the quantum state. The state vector is modified
    in-place for memory efficiency.

    Parameters:
    -----------
    state : complex array, shape (2**n_qubits,)
        Input quantum state vector to be modified in-place
    n_qubits : int
        Number of qubits in the quantum system
    gate_ids : int array, length L
        Array of gate type identifiers (see GATE_* constants)
    wire1 : int array, length L
        Primary qubit indices:
        - For 1-qubit gates: target qubit
        - For 2-qubit gates: control qubit
    wire2 : int array, length L  
        Secondary qubit indices:
        - For 1-qubit gates: -1 (unused)
        - For 2-qubit gates: target qubit
    theta : float array, length L
        Rotation angles in radians:
        - For rotation gates (Rx, Ry, Rz): rotation angle
        - For other gates: ignored
        
    Returns:
    --------
    state : complex array
        The modified state vector (same object as input)
        
    Notes:
    ------
    The function uses a simple switch-case pattern to dispatch to the
    appropriate gate implementation based on the gate ID. Unknown gate
    types are silently ignored (no-op).
    """
    L = gate_ids.shape[0]  # Number of gates in the circuit
    
    # Sequential execution of each gate in the circuit
    for k in range(L):
        g = gate_ids[k]  # Gate type
        a = wire1[k]     # Primary qubit
        b = wire2[k]     # Secondary qubit (if applicable)
        t = theta[k]     # Rotation angle (if applicable)
        
        # Dispatch to appropriate gate implementation
        if g == GATE_X:
            _apply_x(state, n_qubits, a)
        elif g == GATE_Z:
            _apply_z(state, n_qubits, a)
        elif g == GATE_H:
            _apply_h(state, n_qubits, a)
        elif g == GATE_RX:
            _apply_rx(state, n_qubits, a, t)
        elif g == GATE_RY:
            _apply_ry(state, n_qubits, a, t)
        elif g == GATE_RZ:
            _apply_rz(state, n_qubits, a, t)
        elif g == GATE_CX:
            _apply_cx(state, n_qubits, a, b)
        elif g == GATE_CZ:
            _apply_cz(state, n_qubits, a, b)
        else:
            # Unknown gate type: no-op (silently ignore)
            continue

    return state


def run_circuit(n_qubits, gate_ids, wire1, wire2, theta, input_state=None):
    """
    Execute a quantum circuit starting from the |0...0⟩ state.
    
    This is a convenience function that allocates a fresh computational
    basis state |0...0⟩ and then applies the specified circuit.
    
    Parameters:
    -----------
    n_qubits : int
        Number of qubits in the quantum system
    gate_ids : int array
        Gate type identifiers
    wire1 : int array  
        Primary qubit indices
    wire2 : int array
        Secondary qubit indices
    theta : float array
        Rotation angles

    input_state: complex array, shape (2**n_qubits,), optional
        Initial quantum state vector. If None, starts from |0...0⟩.
        
    Returns:
    --------
    out_state : complex array, shape (2**n_qubits,)
        Final quantum state vector after circuit execution
        
    Notes:
    ------
    The initial state is |0...0⟩ = [1, 0, 0, ..., 0] in the computational basis.
    Complex64 is often sufficient for quantum simulations and uses half the memory
    compared to complex128.
    """
    if input_state is None:
        input_state = np.zeros((2**n_qubits,), dtype=np.complex64)
        input_state[0] = 1.0 + 0.0j  # |0...0⟩ state

    run_circuit_inplace(input_state, n_qubits, gate_ids, wire1, wire2, theta)
    return input_state


# ------------------------
# Batched executor (optional)
# ------------------------

@njit(parallel=True)
def run_many_states(n_qubits, gate_ids, wire1, wire2, theta, states_in, states_out):
    """
    Execute the same quantum circuit on a batch of input states in parallel.
    
    This function enables efficient batch processing by applying the same
    circuit to multiple different input states simultaneously. Parallelization
    is achieved using Numba's prange for multi-threading.
    
    Parameters:
    -----------
    n_qubits : int
        Number of qubits in each quantum system
    gate_ids : int array
        Gate type identifiers (same circuit applied to all states)
    wire1 : int array
        Primary qubit indices
    wire2 : int array  
        Secondary qubit indices
    theta : float array
        Rotation angles
    states_in : complex array, shape (B, 2**n_qubits)
        Batch of B input quantum states
    states_out : complex array, shape (B, 2**n_qubits)
        Batch of B output quantum states (modified in-place)
        
    Notes:
    ------
    - The same circuit is applied to all input states
    - Each state is processed independently in parallel
    - Input states are copied locally for thread safety
    - This is particularly useful for variational quantum algorithms
      that need to evaluate circuits on multiple initial states
    - The parallel=True decorator enables automatic parallelization
    """
    B = states_in.shape[0]  # Batch size

    if states_out is None:
        states_out = np.empty_like(states_in)
    if states_in.shape[1] != (1 << n_qubits):
        raise ValueError("states_in has incorrect shape for the given n_qubits")
    if states_out.shape[0] != B:
        raise ValueError("states_out must have the same batch size as states_in")
    
    # Process each state in the batch in parallel
    for b in prange(B):
        # Copy input state to local array (Numba prefers contiguous local arrays)
        s = states_in[b].copy()
        
        # Execute the circuit on this state
        run_circuit_with_state(s, n_qubits, gate_ids, wire1, wire2, theta)
        
        # Store the result
        states_out[b] = s

    return states_out
    

## Circuit Building Utilities

This section provides convenience functions for constructing quantum circuits from high-level Python descriptions.

### Circuit Representation:
Rather than manually constructing the parallel arrays required by the executor, users can specify circuits as lists of tuples with natural syntax:

**Single-qubit gates:**
- `(GATE_H, qubit)` - Hadamard on specified qubit
- `(GATE_X, qubit)` - Pauli-X on specified qubit  
- `(GATE_RX, qubit, angle)` - X-rotation with angle

**Two-qubit gates:**
- `(GATE_CX, control, target)` - CNOT gate
- `(GATE_CZ, control, target)` - Controlled-Z gate

The `build_circuit` function converts these high-level descriptions into the efficient parallel array format required by the Numba-compiled executor functions.

In [96]:
# ------------------------
# Convenience: build circuit arrays from Python list
# ------------------------

def build_circuit(circuit_ops, dtype=np.float32):
    """
    Convert a high-level circuit description into parallel arrays for the executor.
    
    This function provides a user-friendly interface for constructing quantum
    circuits. Instead of manually building the parallel arrays required by the
    Numba-compiled functions, users can specify circuits using intuitive tuples.
    
    Parameters:
    -----------
    ops : list of tuples
        Circuit description as a list of gate operations:
        
        Single-qubit gates (no angle):
        - (GATE_H, q)      : Hadamard gate on qubit q
        - (GATE_X, q)      : Pauli-X gate on qubit q  
        - (GATE_Z, q)      : Pauli-Z gate on qubit q
        
        Single-qubit rotation gates (with angle):
        - (GATE_RX, q, θ)  : X-rotation by angle θ on qubit q
        - (GATE_RY, q, θ)  : Y-rotation by angle θ on qubit q
        - (GATE_RZ, q, θ)  : Z-rotation by angle θ on qubit q
        
        Two-qubit gates:
        - (GATE_CX, c, t)  : CNOT with control c and target t
        - (GATE_CZ, c, t)  : Controlled-Z with control c and target t
        
    dtype : numpy dtype, optional (default=np.float32)
        Data type for the theta array (angles)
        
    Returns:
    --------
    tuple of (gate_ids, wire1, wire2, theta)
        gate_ids : int32 array
            Gate type identifiers
        wire1 : int32 array  
            Primary qubit indices (target for 1q, control for 2q)
        wire2 : int32 array
            Secondary qubit indices (-1 for 1q, target for 2q)
        theta : float array
            Rotation angles (0.0 for non-rotation gates)
            
    Example:
    --------
    >>> ops = [
    ...     (GATE_H, 0),           # Hadamard on qubit 0
    ...     (GATE_CX, 0, 1),       # CNOT: control=0, target=1  
    ...     (GATE_RZ, 1, 0.5),     # Z-rotation by 0.5 radians on qubit 1
    ... ]
    >>> gate_ids, w1, w2, theta = build_circuit(ops)
    
    Notes:
    ------
    - The function validates gate types and raises ValueError for unknown gates
    - All arrays are converted to appropriate NumPy dtypes for Numba compatibility
    - The wire2 array contains -1 for single-qubit gates (unused parameter)
    - The theta array contains 0.0 for non-parameterized gates
    """
    # Initialize lists to collect circuit components
    gate_ids, w1, w2, th = [], [], [], []
    
    # Process each operation in the circuit
    for op in circuit_ops:
        gate, qubits, param = op
        g = GATE_DICT[gate]  # Gate type identifier
        
        
        # Handle single-qubit gates without parameters
        if g in (GATE_X, GATE_Z, GATE_H):
            gate_ids.append(g)
            w1.append(qubits[0])      # Target qubit
            w2.append(-1)         # No second qubit (unused)
            th.append(0.0)        # No angle parameter
            
        # Handle parameterized single-qubit rotation gates  
        elif g in (GATE_RX, GATE_RY, GATE_RZ):
            gate_ids.append(g)
            w1.append(qubits[0])      # Target qubit
            w2.append(-1)         # No second qubit (unused)
            th.append(float(param[0]))  # Rotation angle
            
        # Handle two-qubit controlled gates
        elif g in (GATE_CX, GATE_CZ):
            gate_ids.append(g)
            w1.append(qubits[0])      # Control qubit
            w2.append(qubits[1])      # Target qubit  
            th.append(0.0)        # No angle parameter
            
        else:
            raise ValueError(f"Unknown gate code: {g}")
    
    # Convert lists to NumPy arrays with appropriate dtypes
    return (
        np.asarray(gate_ids, dtype=np.int32),  # Gate identifiers
        np.asarray(w1, dtype=np.int32),        # Primary qubit indices
        np.asarray(w2, dtype=np.int32),        # Secondary qubit indices  
        np.asarray(th, dtype=dtype),           # Rotation angles
    )


def build_noisy_circuit(circuit_ops, x_noise:np.ndarray, z_noise:np.ndarray):
    noisy_circuit_ops = []
    for i, op in enumerate(circuit_ops):
        noisy_circuit_ops.append(op)
        for q in op[1]:
            noisy_circuit_ops.append(('rx', [q], [x_noise[i].item()]))
            noisy_circuit_ops.append(('rz', [q], [z_noise[i].item()]))


    return build_circuit(noisy_circuit_ops)
            


## Example Usage and Testing

This section demonstrates the simulator in action with a sample quantum circuit. The example shows both single-circuit execution and batch processing capabilities.

### Sample Circuit:
The test circuit operates on 5 qubits and includes:
1. **H(0)**: Hadamard gate creating superposition on qubit 0
2. **CX(0,1)**: CNOT gate entangling qubits 0 and 1  
3. **RZ(3, 0.7)**: Z-rotation by 0.7 radians on qubit 3
4. **RX(4, 0.2)**: X-rotation by 0.2 radians on qubit 4
5. **CZ(2,4)**: Controlled-Z gate between qubits 2 and 4
6. **H(1)**: Another Hadamard gate on qubit 1

This circuit demonstrates the full range of supported gate types and creates a complex entangled state suitable for testing the simulator's correctness and performance.

In [97]:
PQC_GATES = ['rz', 'rx', 'rz']
DATA_PATH = '../../nogit/circuit_tokens/no_uncomp/5q_500g_circuit_data/'
GOOD_DATA_PATH = DATA_PATH + 'per_seed_data/'
BAD_DATA_PATH = DATA_PATH + 'poor_fidelity/'
CONFIG_PATH = DATA_PATH + 'config.json'

with open(CONFIG_PATH, 'r') as f:
    CONFIG = json.load(f)


NUM_QUBITS = CONFIG.get("qubits", 3)[0]
NUM_GATES = CONFIG.get("gates", 4)[0] # Multiply by 2 for uncomp gates. 


MAX_CIRCUITS = 1000

print(f'Number of Qubits: {NUM_QUBITS}, Number of Gates: {NUM_GATES}')

Number of Qubits: 5, Number of Gates: 500


In [98]:
import os

all_circuit_tokens = []

for i, filename in enumerate(os.listdir(GOOD_DATA_PATH)):
    if i > 10000:
        break
    with open(GOOD_DATA_PATH + filename, 'r') as f:
        token_dict = json.load(f)
        all_circuit_tokens.append(token_dict['base_circuit_tokens'])
        f.close()

print(f"Number of good data samples: {len(all_circuit_tokens)}")

Number of good data samples: 10001


In [99]:
from tqdm.auto import tqdm

input_states = np.zeros((100, 2**NUM_QUBITS), dtype=np.complex64)
input_states[:,0] = 1
print(input_states)

output_state = np.zeros_like(input_states) 

x_noise = np.ones((NUM_GATES)) * 0.01
z_noise = np.ones((NUM_GATES)) * 0.01



[[1.+0.j 0.+0.j 0.+0.j ... 0.+0.j 0.+0.j 0.+0.j]
 [1.+0.j 0.+0.j 0.+0.j ... 0.+0.j 0.+0.j 0.+0.j]
 [1.+0.j 0.+0.j 0.+0.j ... 0.+0.j 0.+0.j 0.+0.j]
 ...
 [1.+0.j 0.+0.j 0.+0.j ... 0.+0.j 0.+0.j 0.+0.j]
 [1.+0.j 0.+0.j 0.+0.j ... 0.+0.j 0.+0.j 0.+0.j]
 [1.+0.j 0.+0.j 0.+0.j ... 0.+0.j 0.+0.j 0.+0.j]]


In [100]:

for circuit_tokens in tqdm(all_circuit_tokens):
    gate_ids, w1, w2, theta = build_noisy_circuit(circuit_tokens, x_noise, z_noise)
    # out_state = run_circuit(NUM_QUBITS, gate_ids, w1, w2, theta, dtype_is64=True)  # Use complex64 for efficiency
    run_many_states(NUM_QUBITS, gate_ids, w1, w2, theta, input_states, output_state)
    assert np.allclose(np.linalg.norm(output_state, axis=1), 1)
    





  0%|          | 0/10001 [00:00<?, ?it/s]

## Summary and Performance Notes

The quantum state vector simulator successfully executed the test circuit with the following results:

### ✅ **Verification Results:**
- **State normalization**: ≈ 1.0 (preserves quantum probability conservation)
- **Non-zero amplitudes**: 8 out of 32 basis states (demonstrates quantum superposition)
- **Batch consistency**: All identical inputs produced identical outputs
- **Single vs. batch equivalence**: Results match between execution modes

### 🚀 **Performance Characteristics:**
- **Numba JIT compilation**: First execution includes compilation overhead (~1.8s), subsequent runs are much faster
- **Memory efficiency**: In-place operations minimize memory allocation
- **Parallelization**: Batch processing leverages multiple CPU cores
- **Complex precision**: complex64 provides good balance of accuracy and memory usage

### 🔧 **Implementation Highlights:**
- **Bit manipulation**: Efficient state indexing using bitwise operations
- **Little-endian ordering**: Consistent with quantum computing conventions
- **Gate modularity**: Each gate implemented as a separate optimized function
- **Type safety**: Separate compilation paths for different precision levels

This simulator is suitable for medium-scale quantum circuit simulation (up to ~15-20 qubits depending on available memory) and can serve as a foundation for quantum algorithm development and testing.

In [101]:
import sys
sys.path.append('../../')
import jax

from pqcqec.simulate.simulate import run_circuit_with_noise_model, get_input_data
from pqcqec.noise.simple_noise import PennylaneNoisyGates
from pqcqec.training.jax_loss_functions import jax_pure_state_fidelity

no_noise_model = PennylaneNoisyGates(0,0,0,0)
noise_model = PennylaneNoisyGates(0.01, 0.01, 0, 0)


In [102]:

# for i, circuit_tokens in tqdm(enumerate(all_circuit_tokens)):
#     custom_out_state = np.zeros_like(input_states)
#     gate_ids, w1, w2, theta = build_circuit(circuit_tokens)
#     custom_out_state = run_many_states(NUM_QUBITS, gate_ids, w1, w2, theta, input_states, None)

#     pennylane_out_state = run_circuit_with_noise_model(circuit_tokens, input_states, no_noise_model, NUM_QUBITS, batched=True)
#     out_state_fid = jax.vmap(jax_pure_state_fidelity, in_axes=(0,0))(custom_out_state, pennylane_out_state)

#     assert np.allclose(out_state_fid, 1), out_state_fid



In [141]:
nq = 2
ops = [

    ('x', [0], []),
    # ('x', [1], []),
    # ('z', [2], []), 
    # ('cx', [0,1], []),
    # ('cx', [1,0], []),
    
]

gid, w1, w2, p = build_circuit(ops)

# input_states = np.array(get_input_data(nq, 1))
input_states = np.zeros((1, 2**nq,), dtype=np.complex64)
input_states[:, 0] = 1.0 + 0.0j

assert np.allclose(np.linalg.norm(input_states, axis=1), 1)


print(gid)
print(w1)
print(w2)   
print(p)

o1 = run_many_states(nq, gid, w1, w2, p, input_states, None)
print(o1.shape)


[0]
[0]
[-1]
[0.]
(1, 4)


In [142]:

o2 = run_circuit_with_noise_model(ops, input_states, no_noise_model, nq, batched=True)
print(o2.shape)

(1, 4)


In [143]:
probs_1 = np.abs(o1**2)
probs_2 = np.abs(o2**2)

In [144]:
jax.vmap(jax_pure_state_fidelity)(o1, o2)

Array([0.], dtype=float32)

In [145]:
np.allclose(o1, o2)

False

In [146]:
np.allclose(probs_1, probs_2)

False

In [147]:
for x in list(zip(probs_1, probs_2)):
    p1, p2 = x
    assert np.allclose(np.sum(p1), 1)
    assert np.allclose(np.sum(p2), 1)

    if not np.allclose(p1, p2):
        print("Discrepancy found:")
        print(p1)
        print(p2)
        print('---')

Discrepancy found:
[0. 1. 0. 0.]
[0. 0. 1. 0.]
---


In [148]:
print(o1, o2, sep='\n')

[[0.+0.j 1.+0.j 0.+0.j 0.+0.j]]
[[0.+0.j 0.+0.j 1.+0.j 0.+0.j]]
